# Módulo 5 — Eficiencia: cuantización int4 (nf4) del generador

> **Corre en Colab con runtime T4 GPU.** Este notebook se entrega con las
> salidas vacías. Ejecuta de arriba a abajo; los números los produce la
> corrida T4 y se pegan en `final/efficiency/RESULTS.md`. **No hay métricas
> inventadas.**

Aplica int4 (nf4, QLoRA-style) al componente más lento del sistema (el
generador Qwen3-1.7B + adaptador LoRA de M1) y produce la tabla
antes/después sobre el **mismo** eval set y el **mismo** harness congelado,
más las pruebas que no miden exactitud (invariancia, válvula, adversarial)
sobre la versión cuantizada.

## 0 · Setup: clonar el repo e instalar deps (igual que M3)

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/iamcroody/JoseJimenez-JuanYoung-AlbertoCervantes-DavidCuadros.git"
# La rama de trabajo del Módulo 5: trae final/efficiency/ y este notebook.
BRANCH = "m5-final-integrator"
ROOT = Path("/content/JoseJimenez-JuanYoung-AlbertoCervantes-DavidCuadros")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--recurse-submodules", "-b", BRANCH,
                        REPO, str(ROOT)], check=True)
    os.chdir(ROOT)
else:
    # Local: sitúate en la raíz del repo (donde existe eval/).
    ROOT = Path.cwd() if (Path.cwd() / "eval").exists() else Path.cwd().parent
    os.chdir(ROOT)
print("cwd:", Path.cwd(), "| branch:", BRANCH, "| colab:", IN_COLAB)

In [ ]:
# Versiones exactas (bitsandbytes + el stack de m3), para que el índice y la
# tabla reproduzcan. torchao se quita como en M1/M2/M3: Colab lo fija a su
# build de torch y peft lo rechaza.
%pip install -q -r final/efficiency/requirements.txt
%pip uninstall -y -q torchao

import importlib.metadata as mdl
VERSIONS = {p: mdl.version(p) for p in
            ("torch", "transformers", "peft", "sentence-transformers",
             "chromadb", "rouge_score", "bitsandbytes")}
print(VERSIONS)
print("si Colab pide reiniciar el runtime, reinicia y re-ejecuta desde la primera celda")

In [ ]:
import gc, json
import torch
from transformers import set_seed

for path in ("scripts", "eval", "m3", "final/efficiency"):
    if path not in sys.path:
        sys.path.insert(0, path)

import macgyver as mg
import harness as H
import corpus as C
import rag as R
import systems as S

import efficiency as E
import measure as MEA
import invariance as INV

set_seed(H.SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| seed:", H.SEED, "| rubric:", H.load_rubric()["version"])
assert DEVICE == "cuda", "La carga en 4 bits necesita una GPU T4; cambia el runtime a GPU."

REPORTS = Path("reports"); REPORTS.mkdir(exist_ok=True)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 1 · Self-tests sin GPU

Antes de cargar nada: la plomería de eficiencia (percentiles, perturbación,
esquema del dict) y el self-test de M3. Encontrar un error aquí cuesta un
segundo, no una sesión de T4.

In [ ]:
# Self-test de eficiencia (solo numpy) como subproceso, y el self-test de M3.
import subprocess, sys as _sys
res = subprocess.run([_sys.executable, "final/efficiency/selftest.py"],
                     capture_output=True, text=True)
print(res.stdout)
assert res.returncode == 0, res.stderr

import selftest as m3_selftest
m3_selftest.main()

## 2 · Construir el recuperador de M3 (una vez)

Reusa `m3/corpus.py` + `m3/rag.py` con las mismas decisiones del run final de
M3 (embedding + top_k + reranker). El recuperador es compartido por las dos
versiones del generador, así la única variable entre filas es la cuantización.

In [ ]:
eval_set = H.load_eval_set()
docs = C.build_documents()
BY_ID = {d["id"]: d for d in docs}
print(f"{len(eval_set)} casos de eval, {len(docs)} documentos de catálogo")

# Mismas decisiones del run final de M3 (ver notebooks/M3_rag.ipynb).
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
TOP_K = 5
dense = R.DenseIndex(docs, EMBED_MODEL, device=DEVICE)
reranker = R.load_reranker(device=DEVICE)
retriever = R.Retriever(docs, dense, reranker=reranker, k=TOP_K, pool=20)
ADVANCED = R.TECHNIQUES  # ('hybrid','rerank','hyde')
PROMPT_VERSION = S.PROMPT_VERSION  # 'v2a'
print("retriever listo:", EMBED_MODEL, "k=", TOP_K, "| técnicas:", ADVANCED)

## 3 · Fila **Antes**: generador fp16 (la línea base de M3)

El mismo generador que M3 usó en el scorecard: base fp16 con el adaptador LoRA
fusionado. Se liga con `systems.bind` y se mide `sistema_rag_avanzado`.

In [ ]:
# Juez y similitud se cargan una vez y se reusan en las dos filas.
similarity = H.Similarity()
judge = H.Judge()

tok = mg.load_tokenizer()
model_fp16 = E.load_fp16_model()
gen_fp16 = E.make_generator(model_fp16, tok)

S.bind(retriever, gen_fp16, docs, techniques=ADVANCED, prompt_version=PROMPT_VERSION)
before = MEA.measure_system("antes-fp16", S.sistema_rag_avanzado, model_fp16,
                            eval_set, judge=judge, similarity=similarity, verbose=True)
print(json.dumps({k: before[k] for k in
                  ("latency_p50_s", "latency_p95_s", "peak_vram_mb", "footprint_mb")},
                 indent=2))

del model_fp16, gen_fp16
free_gpu()

## 4 · Fila **Después**: generador int4 (nf4)

Base cargado en 4 bits (nf4, doble cuant., cómputo fp16) con el MISMO adaptador
LoRA encima (sin fusionar). El pipeline de RAG no cambia.

In [ ]:
model_int4 = E.load_int4_model()
gen_int4 = E.make_generator(model_int4, tok)

S.bind(retriever, gen_int4, docs, techniques=ADVANCED, prompt_version=PROMPT_VERSION)
after = MEA.measure_system("despues-int4", S.sistema_rag_avanzado, model_int4,
                           eval_set, judge=judge, similarity=similarity, verbose=True)
print(json.dumps({k: after[k] for k in
                  ("latency_p50_s", "latency_p95_s", "peak_vram_mb", "footprint_mb")},
                 indent=2))

## 5 · Pruebas que NO miden exactitud, sobre int4

Invariancia (fracción de respuestas idénticas bajo perturbación trivial),
válvula de escape (adv01/adv04 disparan el rechazo que el parser reconoce) y
bloque adversarial (adv01..adv04 re-puntuados con la dimensión 3). El sistema
int4 sigue ligado de la celda anterior.

In [ ]:
inv = INV.invariance_probe(eval_set, S.sistema_rag_avanzado, verbose=True)
print("invariancia:", inv["invariance_rate"], f"({inv['unchanged']}/{inv['n']})")

valve = INV.valve_check(eval_set, S.sistema_rag_avanzado, verbose=True)
print("válvula dispara en todas (adv01, adv04):", valve["all_fired"])

adv = INV.adversarial_check(eval_set, S.sistema_rag_avanzado, verbose=True)
print("hit-rate adversarial int4:", adv["adversarial_hit_rate"])

## 6 · Escribir reportes y la tabla antes/después

Escribe `reports/efficiency_m5.csv` y `reports/efficiency_m5_detail.json`, y
rellena la tabla de `final/efficiency/RESULTS.md`. Compara contra el
presupuesto de pérdida aceptable fijado **antes** de medir.

In [ ]:
# Adjunta las pruebas no-exactitud al detalle de la fila int4.
after["non_accuracy"] = {"invariance": inv, "valve": valve, "adversarial": adv}

csv_path, json_path = MEA.write_outputs([before, after],
                                        REPORTS / "efficiency_m5.csv",
                                        REPORTS / "efficiency_m5_detail.json")
print("escrito:", csv_path, json_path)
print(Path(csv_path).read_text())

In [ ]:
# Presupuesto: <= 5 puntos de dim3 perdidos (>= 37.9%), VRAM menor y p95 no mayor.
q_before = before["quality"]["domain_hit_rate"]
q_after = after["quality"]["domain_hit_rate"]
dim3_drop_pts = None if (q_before is None or q_after is None) else round((q_before - q_after) * 100, 2)
vram_lower = (before["peak_vram_mb"] is not None and after["peak_vram_mb"] is not None
              and after["peak_vram_mb"] < before["peak_vram_mb"])
p95_not_worse = (before["latency_p95_s"] is not None and after["latency_p95_s"] is not None
                 and after["latency_p95_s"] <= before["latency_p95_s"])
budget_held = (dim3_drop_pts is not None and dim3_drop_pts <= 5 and vram_lower and p95_not_worse)
print(f"dim3: antes={q_before} despues={q_after} (caída {dim3_drop_pts} pts)")
print(f"VRAM menor: {vram_lower} | p95 no peor: {p95_not_worse}")
print(f"¿presupuesto cumplido?  {budget_held}")

In [ ]:
# Rellena la tabla de RESULTS.md con los números de esta corrida.
def _fmt_q(q):
    return "n/a" if q is None else f"{q*100:.1f}%"
def _fmt_s(s):
    return "n/a" if s is None else f"{s:.2f}s"
def _fmt_mb(m):
    return "n/a" if m is None else f"{m:.0f} MB"

row_before = (f"| Antes (M4/M3 fp16) | {_fmt_q(q_before)} | "
              f"{_fmt_s(before['latency_p50_s'])} | {_fmt_s(before['latency_p95_s'])} | "
              f"VRAM {_fmt_mb(before['peak_vram_mb'])} / pesos {_fmt_mb(before['footprint_mb'])} |")
row_after = (f"| Despues (int4 nf4) | {_fmt_q(q_after)} | "
             f"{_fmt_s(after['latency_p50_s'])} | {_fmt_s(after['latency_p95_s'])} | "
             f"VRAM {_fmt_mb(after['peak_vram_mb'])} / pesos {_fmt_mb(after['footprint_mb'])} |")
print(row_before)
print(row_after)
print(f"\n¿Se cumplió el presupuesto?  {'SÍ' if budget_held else 'NO'}")
print("Pega estas filas y la respuesta del presupuesto en final/efficiency/RESULTS.md,")
print("junto con invariancia=", inv['invariance_rate'],
      "válvula=", valve['all_fired'], "adversarial=", adv['adversarial_hit_rate'])